# Proyecto Final: Clasificación Multiclase y Triaje de Radiografías de Tórax mediante Aprendizaje Profundo

**Autor:** David Kira  
**Asignatura:** Bioinformática (4.° Año — UNSTA)  
**Entorno de trabajo:** Google Colaboratory sobre GPU NVIDIA T4 con PyTorch (Paszke et al., 2019).  

---

## 1. Origen de los Datos y Contexto Experimental

link del dataset: https://nihcc.app.box.com/v/ChestXray-NIHCC

El material radiográfico empleado en este trabajo proviene de la base de datos pública **ChestX-ray8** (ampliada luego como *ChestX-ray14*), provista por el **National Institutes of Health (NIH) Clinical Center** (2017) y presentada por investigadores del *Department of Radiology and Imaging Sciences* del NIH Clinical Center y del *National Center for Biotechnology Information (NCBI)* en Bethesda, Maryland, Estados Unidos:

> Wang, X., Peng, Y., Lu, L., Lu, Z., Bagheri, M., & Summers, R. M. (2017). ChestX-ray8: Hospital-scale chest X-ray database and benchmarks on weakly-supervised classification and localization of common thorax diseases. En *Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition (CVPR)* (pp. 2097–2106). IEEE. https://doi.org/10.1109/CVPR.2017.369

El objetivo de Wang et al. (2017) fue construir un repositorio de radiografías de tórax a escala hospitalaria que permitiera entrenar modelos de aprendizaje profundo sin depender de la anotación manual placa por placa. Para ello, el equipo del NIH recopiló desde el sistema PACS (*Picture Archiving and Communication Systems*) institucional 108.948 radiografías frontales —en proyecciones posteroanterior (PA) y anteroposterior (AP)— correspondientes a 32.717 pacientes (la revisión `v2020` del catálogo extiende el registro a 112.120 imágenes de 30.805 pacientes; National Institutes of Health Clinical Center, 2017). Las etiquetas diagnósticas se obtuvieron procesando de forma automática el texto de los informes radiológicos mediante las herramientas *DNorm* y *MetaMap*, aplicando reglas sintácticas para identificar hallazgos afirmativos y descartar menciones negadas o inciertas.

Cuando el informe no registraba ninguna de las patologías torácicas rastreadas por el sistema de minería de texto, la radiografía recibía la etiqueta **`No Finding` (`Sin hallazgos`)**. Es importante precisar el alcance de esta categoría: `No Finding` indica únicamente que no se detectaron las afecciones específicas contempladas en el catálogo de *ChestX-ray8/14*, pero no garantiza que la radiografía sea completamente normal ni que el paciente esté sano, dado que pueden existir otras alteraciones no incluidas en el diccionario de extracción, dispositivos médicos o hallazgos omitidos por el procesamiento automático del texto (Wang et al., 2017).

Sobre esa base, Wang et al. (2017) establecieron las primeras métricas de referencia (*benchmarks*) de clasificación y localización débilmente supervisada adaptando redes convolucionales preentrenadas en *ImageNet* (*AlexNet*, *GoogLeNet*, *VGGNet-16* y *ResNet-50*). Por su parte, la arquitectura de redes convolucionales densamente conectadas (**DenseNet-121**; Huang et al., 2017) permitió reutilizar los mapas de activación de todas las capas precedentes dentro de cada bloque, reduciendo el número de parámetros y facilitando la transferencia de aprendizaje en el análisis de imágenes médicas (Litjens et al., 2017).

El uso de etiquetas generadas automáticamente plantea dos problemas metodológicos para un clasificador multiclase (Wang et al., 2017). El primero es la **co-ocurrencia multietiqueta**: en la práctica hospitalaria, cuadros como la insuficiencia cardíaca descompensada suelen presentar agrandamiento cardíaco y acumulación de líquido pleural en simultáneo, por lo que abundan los registros etiquetados como `Cardiomegaly|Effusion`. Si un modelo de clases mutuamente excluyentes intenta separar ambas afecciones sin filtrar antes esos casos mixtos, recibe supervisión contradictoria. El segundo problema deriva del seguimiento longitudinal de los pacientes internados, quienes cuentan con múltiples radiografías sucesivas; repartir estudios de un mismo individuo entre los subconjuntos de entrenamiento y prueba introduce fuga de información (*data leakage*), ya que la red puede memorizar rasgos anatómicos del paciente en lugar del signo radiológico.




## 2. Objetivos del Trabajo

Este trabajo implementa y evalúa un clasificador experimental de radiografías de tórax basado en **DenseNet-121** (Huang et al., 2017) sobre tres categorías del catálogo del NIH Clinical Center (2017): **Sin hallazgos (`0`, `No Finding`)**, **Cardiomegalia (`1`, `Cardiomegaly`)** y **Derrame Pleural (`2`, `Effusion`)**.

El análisis persigue cuatro objetivos específicos:
1. Depurar un subconjunto de datos a partir de `Data_Entry_2017_v2020.csv` eliminando la co-ocurrencia entre `Cardiomegaly` y `Effusion` y asegurando que cada `Patient_ID` conserve una única categoría asignada en todo el registro.
2. Describir estadísticamente el conjunto resultante siguiendo la estructura analítica del *Mini-Proyecto 2* (variables, dimensiones tensoriales, control de valores nulos, proporciones por clase y frecuencia de estudios por paciente) y dividir la muestra mediante particionado estratificado por grupos (`StratifiedGroupKFold`; Pedregosa et al., 2011) para evitar la fuga de datos entre pacientes.
3. Entrenar la arquitectura `DenseNet-121` (Huang et al., 2017) en PyTorch (Paszke et al., 2019) mediante ajuste fino (*fine-tuning*) de sus bloques convolucionales profundos durante 40 épocas con regularización explícita (`Dropout`, decaimiento $L_2$ y *label smoothing*).
4. Evaluar el desempeño del modelo sobre el conjunto de prueba tanto en clasificación multiclase directa como en un esquema de dos etapas —una primera etapa de triaje binario (`Sin hallazgos` vs. `Patología detectada`) con umbrales ajustados en validación, y una segunda etapa condicional de diferenciación entre `Cardiomegalia` y `Derrame Pleural` sobre los casos patológicos—, analizando su alcance y limitaciones.


## Parte 01: Selección de Muestras y Preprocesamiento de Imágenes

El catálogo original de *ChestX-ray14* supera los 40 GB distribuidos en doce volúmenes `.tar.gz` (National Institutes of Health Clinical Center, 2017). Para procesar una muestra manejable en Google Colaboratory, el siguiente bloque filtra el archivo `Data_Entry_2017_v2020.csv` y selecciona 800 radiografías para cada una de las tres clases (`Sin hallazgos`, `Cardiomegalia` y `Derrame Pleural`).

La función `seleccionar_cohorte_balanceada()` exige que los registros de `Cardiomegaly` no contengan la cadena `Effusion` en `Finding Labels`, y viceversa, cruzando además los identificadores `Patient ID` para que ningún paciente figure en más de una categoría. Luego verifica que cada subconjunto filtrado disponga de al menos `n_por_clase` registros y aplica muestreo aleatorio reproducible (`.sample(n=n_por_clase, random_state=42)`) en las tres clases. Una vez descomprimidas las placas seleccionadas, la rutina convierte cada imagen a formato `RGB` de tres canales y reduce su resolución de $1024 \times 1024$ a **$224 \times 224$ píxeles** con interpolación de Lanczos, sin aplicar filtro de mediana previo para no atenuar el reborde costal ni la línea pleural.



In [ ]:
import glob
import os
import tarfile
import zipfile
from pathlib import Path

import pandas as pd
from google.colab import drive
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

RUTA_PROYECTO = Path('/content/drive/MyDrive/Unsta/4to año/Proyecto final - Bioinformatica - Procesamiento de imagenes/Dataset/CXR8')
RUTA_CSV_NIH = RUTA_PROYECTO / 'Data_Entry_2017_v2020.csv'
DIR_VOLUMENES_TAR = RUTA_PROYECTO / 'images'

RUTA_ZIP_SALIDA = Path('/content/drive/MyDrive/multiclase_dataset.zip')
RUTA_CSV_SALIDA = Path('/content/drive/MyDrive/multiclase_labels.csv')

DIR_TRABAJO = Path('/content/CXR8_local')
DIR_CRUDAS = DIR_TRABAJO / 'raw'
DIR_PROCESADAS = DIR_TRABAJO / 'processed'


def seleccionar_cohorte_balanceada(
    catalogo: pd.DataFrame, n_por_clase: int = 800, semilla: int = 42
) -> pd.DataFrame:
    """Selecciona muestras aleatorias de Sin hallazgos, Cardiomegalia y Derrame sin solapamiento."""
    etiquetas = catalogo['Finding Labels']

    sin_hallazgos = catalogo[etiquetas == 'No Finding'].copy()
    cardiomegalia = catalogo[
        etiquetas.str.contains('Cardiomegaly', na=False)
        & ~etiquetas.str.contains('Effusion', na=False)
    ].copy()
    derrame = catalogo[
        etiquetas.str.contains('Effusion', na=False)
        & ~etiquetas.str.contains('Cardiomegaly', na=False)
    ].copy()

    # Se restringe cada Patient ID a una sola categoría para evitar diagnósticos cruzados
    # entre estudios longitudinales de un mismo paciente.
    pacientes_cardio = set(cardiomegalia['Patient ID'].unique())
    derrame = derrame[~derrame['Patient ID'].isin(pacientes_cardio)].copy()
    pacientes_derrame = set(derrame['Patient ID'].unique())
    sin_hallazgos = sin_hallazgos[
        ~sin_hallazgos['Patient ID'].isin(pacientes_cardio | pacientes_derrame)
    ].copy()

    candidatos = [
        ('Sin hallazgos (0)', sin_hallazgos, 0),
        ('Cardiomegalia (1)', cardiomegalia, 1),
        ('Derrame Pleural (2)', derrame, 2),
    ]
    for nombre_clase, df_clase, _ in candidatos:
        if len(df_clase) < n_por_clase:
            raise ValueError(
                f'Registros insuficientes en {nombre_clase}: '
                f'disponibles={len(df_clase)}, requeridos={n_por_clase}.'
            )

    muestras = [
        df_clase.sample(n=n_por_clase, random_state=semilla).assign(Label=codigo)
        for _, df_clase, codigo in candidatos
    ]
    cohorte = pd.concat(muestras, ignore_index=True)
    return cohorte.sample(frac=1.0, random_state=semilla).reset_index(drop=True)


def extraer_placas_desde_tar(archivos_tar: list[str], nombres_objetivo: set[str], destino: Path) -> None:
    pendientes = set(nombres_objetivo)
    for ruta_tar in archivos_tar:
        if not pendientes:
            break
        with tarfile.open(ruta_tar, 'r:gz') as archivo_tar:
            for miembro in archivo_tar.getmembers():
                nombre = os.path.basename(miembro.name)
                if nombre in pendientes:
                    miembro.name = nombre
                    archivo_tar.extract(miembro, path=destino)
                    pendientes.remove(nombre)


def normalizar_resolucion_placas(
    cohorte: pd.DataFrame, dir_origen: Path, dir_destino: Path, lado_px: int = 224
) -> pd.DataFrame:
    registros_validos = []
    for _, fila in tqdm(cohorte.iterrows(), total=len(cohorte), desc='Reescalando radiografías'):
        nombre_png = fila['Image Index']
        ruta_png = dir_origen / nombre_png
        if not ruta_png.exists():
            continue

        nombre_jpg = f"{Path(nombre_png).stem}.jpg"
        ruta_jpg = dir_destino / nombre_jpg
        try:
            with Image.open(ruta_png) as img:
                img_rgb = img.convert('RGB').resize((lado_px, lado_px), Image.Resampling.LANCZOS)
                img_rgb.save(ruta_jpg, 'JPEG', quality=95)
        except (UnidentifiedImageError, OSError) as err:
            print(f'Archivo omitido por lectura fallida ({nombre_png}): {err}')
            continue

        registros_validos.append({
            'Image_Name': nombre_jpg,
            'Patient_ID': int(fila['Patient ID']),
            'Original_Labels': fila['Finding Labels'],
            'Label': int(fila['Label']),
        })
    return pd.DataFrame(registros_validos)


if not RUTA_ZIP_SALIDA.exists() or not RUTA_CSV_SALIDA.exists():
    DIR_CRUDAS.mkdir(parents=True, exist_ok=True)
    DIR_PROCESADAS.mkdir(parents=True, exist_ok=True)

    catalogo_nih = pd.read_csv(RUTA_CSV_NIH)
    cohorte_objetivo = seleccionar_cohorte_balanceada(catalogo_nih, n_por_clase=800, semilla=42)

    volumenes = sorted(glob.glob(str(DIR_VOLUMENES_TAR / '*.tar.gz')))
    extraer_placas_desde_tar(volumenes, set(cohorte_objetivo['Image Index']), DIR_CRUDAS)

    catalogo_procesado = normalizar_resolucion_placas(cohorte_objetivo, DIR_CRUDAS, DIR_PROCESADAS)

    with zipfile.ZipFile(RUTA_ZIP_SALIDA, 'w', zipfile.ZIP_DEFLATED) as archivo_zip:
        for ruta_jpg in DIR_PROCESADAS.glob('*.jpg'):
            archivo_zip.write(ruta_jpg, arcname=ruta_jpg.name)

    catalogo_procesado.to_csv(RUTA_CSV_SALIDA, index=False)
    print(f'Extracción finalizada: {len(catalogo_procesado)} placas guardadas en {RUTA_ZIP_SALIDA}')
else:
    print(f'Artefactos ya compilados en Drive: {RUTA_CSV_SALIDA.name} y {RUTA_ZIP_SALIDA.name}')

---
## Parte 02: Exploración Descriptiva del Dataset y Depuración de Etiquetas

Con el paquete de imágenes descomprimido en el entorno local (`/content/CXR8_local/processed`), el siguiente bloque examina el archivo `multiclase_labels.csv` siguiendo la estructura descriptiva del *Mini-Proyecto 2*: formato de columnas, tipos de datos, verificación de valores faltantes, limpieza de registros con patologías cruzadas y distribución de muestras por clase y por paciente.

In [ ]:
import os
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from google.colab import drive
from PIL import Image

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

RUTA_CSV_CRUDO = Path('/content/drive/MyDrive/multiclase_labels.csv')
RUTA_CSV_LIMPIO = Path('/content/drive/MyDrive/multiclase_labels_limpio.csv')
RUTA_ZIP = Path('/content/drive/MyDrive/multiclase_dataset.zip')
DIR_PLACAS = Path('/content/CXR8_local/processed')

NOMBRES_CLASE = {0: 'Sin hallazgos (0)', 1: 'Cardiomegalia (1)', 2: 'Derrame Pleural (2)'}


def depurar_catalogo_radiologico(catalogo_crudo: pd.DataFrame, dir_imagenes: Path) -> pd.DataFrame:
    """Elimina placas mixtas Cardiomegaly|Effusion y pacientes con diagnósticos múltiples."""
    catalogo = catalogo_crudo.copy()
    catalogo['full_path'] = catalogo['Image_Name'].apply(lambda nombre: str(dir_imagenes / nombre))

    # En la extracción inicial del archivo multiclase_labels.csv se filtró Cardiomegaly
    # dentro de Derrame, pero no Effusion dentro de Cardiomegalia; se depuran aquí esos registros.
    coocurrencia_derrame = (catalogo['Label'] == 1) & (
        catalogo['Original_Labels'].str.contains('Effusion', na=False)
    )
    catalogo = catalogo[~coocurrencia_derrame].copy()

    clases_por_paciente = catalogo.groupby('Patient_ID')['Label'].nunique()
    ids_inconsistentes = clases_por_paciente[clases_por_paciente > 1].index
    catalogo = catalogo[~catalogo['Patient_ID'].isin(ids_inconsistentes)].reset_index(drop=True)

    print(
        f'Depuración completada: {int(coocurrencia_derrame.sum())} placas mixtas y '
        f'{len(ids_inconsistentes)} pacientes con etiquetas múltiples descartados. '
        f'Total limpio: {len(catalogo)} radiografías.'
    )
    return catalogo


def graficar_distribucion_cohorte(catalogo: pd.DataFrame) -> None:
    fig, (ax_clases, ax_pacientes) = plt.subplots(1, 2, figsize=(13, 4.4))
    paleta = ['#2ca02c', '#1f77b4', '#d62728']

    conteo = catalogo['Label'].value_counts().sort_index()
    etiquetas = [NOMBRES_CLASE[i] for i in conteo.index]
    ax_clases.bar(etiquetas, conteo.values, color=paleta, edgecolor='black', alpha=0.85)
    for idx, total in enumerate(conteo.values):
        pct = total / len(catalogo) * 100
        ax_clases.text(idx, total + 12, f'{total} ({pct:.1f}%)', ha='center', fontweight='bold')

    ax_clases.set_ylim(0, conteo.max() * 1.18)
    ax_clases.set_title('Radiografías por Categoría (Catálogo Depurado)')
    ax_clases.set_ylabel('Número de placas')
    ax_clases.grid(axis='y', linestyle='--', alpha=0.4)

    estudios_por_paciente = catalogo.groupby('Patient_ID').size()
    bins_estudios = range(1, int(estudios_por_paciente.max()) + 2)
    ax_pacientes.hist(
        estudios_por_paciente, bins=bins_estudios, color='#9467bd', edgecolor='black', align='left', rwidth=0.8
    )
    ax_pacientes.set_title('Frecuencia de Estudios por Paciente')
    ax_pacientes.set_xlabel('Placas asociadas a un mismo Patient_ID')
    ax_pacientes.set_ylabel('Pacientes (escala log)')
    ax_pacientes.set_yscale('log')
    ax_pacientes.grid(axis='y', linestyle='--', alpha=0.4)

    plt.tight_layout()
    plt.show()


def mostrar_ejemplos_por_clase(catalogo: pd.DataFrame) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
    for codigo, nombre in NOMBRES_CLASE.items():
        ejemplo = catalogo[catalogo['Label'] == codigo].iloc[0]
        ruta_img = Path(ejemplo['full_path'])
        if ruta_img.exists():
            with Image.open(ruta_img) as img:
                axes[codigo].imshow(img, cmap='gray')
        axes[codigo].set_title(f"{nombre}\nPatient_ID: {ejemplo['Patient_ID']}", fontsize=10)
        axes[codigo].axis('off')

    plt.suptitle('Muestras Preprocesadas (224x224 px)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()


if not DIR_PLACAS.exists() or not any(DIR_PLACAS.iterdir()):
    DIR_PLACAS.mkdir(parents=True, exist_ok=True)
    if RUTA_ZIP.exists():
        with zipfile.ZipFile(RUTA_ZIP, 'r') as zf:
            zf.extractall(DIR_PLACAS)

catalogo_inicial = pd.read_csv(RUTA_CSV_CRUDO)
print(
    f'Catálogo inicial: {catalogo_inicial.shape[0]} filas x {catalogo_inicial.shape[1]} columnas | '
    f'Valores nulos: {int(catalogo_inicial.isnull().sum().sum())}'
)
display(catalogo_inicial.head())

df_dataset = depurar_catalogo_radiologico(catalogo_inicial, DIR_PLACAS)

columnas_exportables = ['Image_Name', 'Patient_ID', 'Original_Labels', 'Label']
df_dataset[columnas_exportables].to_csv(RUTA_CSV_LIMPIO, index=False)
df_dataset[columnas_exportables].to_csv('multiclase_labels_limpio.csv', index=False)

tabla_comparativa = pd.DataFrame({
    'Muestras Iniciales': catalogo_inicial['Label'].map(NOMBRES_CLASE).value_counts(),
    'Muestras Depuradas': df_dataset['Label'].map(NOMBRES_CLASE).value_counts(),
    'Proporción Final (%)': (df_dataset['Label'].map(NOMBRES_CLASE).value_counts(normalize=True) * 100).round(2),
})
display(tabla_comparativa)

graficar_distribucion_cohorte(df_dataset)
mostrar_ejemplos_por_clase(df_dataset)

### Análisis de las Variables y Estadísticas Generales

#### Estructura del catálogo y significado de las columnas
El archivo `multiclase_labels.csv` consta de **4 columnas** sin valores faltantes (`0 NaN`):

| Columna | Tipo (`dtype`) | Función | Descripción |
| :--- | :---: | :---: | :--- |
| **`Image_Name`** | `object` | Identificador de imagen | Nombre del archivo `.jpg` redimensionado a $224 \times 224$ px. |
| **`Patient_ID`** | `int64` | Agrupador clínico | Identificador anonimizado del paciente en el catálogo del NIH (National Institutes of Health Clinical Center, 2017). |
| **`Original_Labels`** | `object` | Etiqueta cruda (NLP) | Hallazgos extraídos del informe original en Wang et al. (2017), concatenados con `|`. |
| **`Label`** | `int64` | Variable objetivo | Categoría numérica asignada: `0` (`Sin hallazgos` / `No Finding`), `1` (`Cardiomegalia`) o `2` (`Derrame Pleural`). |

Respecto a la etiqueta `0` (**`Sin hallazgos`**), cabe reiterar que representa los estudios codificados como `No Finding` en `Original_Labels`: indica que el procesamiento automático del informe radiológico no registró ninguna de las 14 patologías del catálogo *ChestX-ray14*, sin implicar necesariamente la ausencia absoluta de cualquier otra anomalía torácica no relevada en dicha lista (Wang et al., 2017).

En cuanto a las variables de entrada (`X`), cada imagen constituye un arreglo de **$224 \times 224 \times 3 = 150.528$ valores numéricos**. En el archivo JPEG original, los píxeles son enteros de 8 bits en el rango **$[0, 255]$**. Durante la carga en PyTorch (Paszke et al., 2019), `ToTensor()` los escala al intervalo continuo **$[0.0, 1.0]$** (`float32`) y `Normalize()` los estandariza según la media y desviación estándar de *ImageNet*, dejando los valores dentro del intervalo aproximado **$[-2.12, +2.64]$**.

#### Efecto de la depuración sobre el tamaño y balance del conjunto de datos
El archivo `multiclase_labels.csv` compilado inicialmente reunía 2.400 placas (800 por categoría). Sin embargo, la inspección de la columna `Original_Labels` reveló que, durante esa primera extracción, se había excluido `Cardiomegaly` al conformar la clase `Derrame Pleural`, pero no se había excluido `Effusion` al conformar `Cardiomegalia`. Como resultado, más de trescientas radiografías asignadas a la clase `1` presentaban derrame pleural concomitante (`Cardiomegaly|Effusion`), además de existir pacientes con distintos estudios repartidos entre categorías.

Tras aplicar `depurar_catalogo_radiologico()`, el conjunto limpio (`multiclase_labels_limpio.csv`) conserva **1.739 radiografías** correspondientes a **1.518 pacientes únicos**: **759 de `Sin hallazgos (0)` ($43.6\%$)**, **407 de `Cardiomegalia (1)` ($23.4\%$)** y **573 de `Derrame Pleural (2)` ($33.0\%$)**. La razón entre la categoría más frecuente y la menos frecuente pasa a ser de $1.86 : 1$. Esta diferencia de tamaños, junto con el hecho de que las dos clases patológicas suman el $56.4\%$ del total ($980$ placas con hallazgos patológicos frente a $759$ sin hallazgos), justifica ponderar la función de pérdida por clase durante el entrenamiento.


## Parte 03: División de Datos Estratificada por Paciente (`StratifiedGroupKFold`)

El histograma de la sección anterior muestra que varios pacientes poseen más de una radiografía en el catálogo. Si la división entre entrenamiento y prueba se realizara al azar por imagen individual, distintos estudios de una misma persona quedarían repartidos en ambos subconjuntos. Por otra parte, un divisor por grupos no estratificado como `GroupShuffleSplit` respeta la agrupación por `Patient_ID`, pero no controla la proporción de clases en cada pliegue.

Para satisfacer ambas condiciones simultáneamente, el siguiente bloque utiliza `StratifiedGroupKFold` con $k = 7$ particiones (Pedregosa et al., 2011): cinco pliegues conforman el conjunto de entrenamiento ($\sim 71.4\%$), uno el de validación ($\sim 14.3\%$) y el restante el de prueba ($\sim 14.3\%$), verificando explícitamente que no existan identificadores `Patient_ID` compartidos entre los tres subconjuntos.

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold


def particionar_por_paciente(
    catalogo: pd.DataFrame, n_particiones: int = 7, semilla: int = 42
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Divide en entrenamiento (5/7), validación (1/7) y prueba (1/7) agrupando por Patient_ID."""
    divisor = StratifiedGroupKFold(n_splits=n_particiones, shuffle=True, random_state=semilla)
    pliegues = list(divisor.split(catalogo, y=catalogo['Label'], groups=catalogo['Patient_ID']))

    idx_val = pliegues[0][1]
    idx_test = pliegues[1][1]
    idx_train = [idx for i in range(2, n_particiones) for idx in pliegues[i][1]]

    sub_train = catalogo.iloc[idx_train].reset_index(drop=True)
    sub_val = catalogo.iloc[idx_val].reset_index(drop=True)
    sub_test = catalogo.iloc[idx_test].reset_index(drop=True)

    ids_train = set(sub_train['Patient_ID'])
    ids_val = set(sub_val['Patient_ID'])
    ids_test = set(sub_test['Patient_ID'])
    if not (ids_train.isdisjoint(ids_val) and ids_train.isdisjoint(ids_test) and ids_val.isdisjoint(ids_test)):
        raise RuntimeError('Solapamiento de Patient_ID detectado entre particiones.')

    return sub_train, sub_val, sub_test


df_train, df_val, df_test = particionar_por_paciente(df_dataset)

for nombre_particion, subconjunto in [('Train', df_train), ('Val  ', df_val), ('Test ', df_test)]:
    reparto = subconjunto['Label'].value_counts().sort_index().to_dict()
    n_pacientes = subconjunto['Patient_ID'].nunique()
    print(f'{nombre_particion}: {len(subconjunto):4d} placas ({n_pacientes:4d} pacientes) -> {reparto}')

Train: 1241 placas (1084 pacientes) -> {0: 541, 1: 291, 2: 409}
Val  :  249 placas ( 217 pacientes) -> {0: 109, 1: 58, 2: 82}
Test :  249 placas ( 217 pacientes) -> {0: 109, 1: 58, 2: 82}


### Balance Estadístico y Aislamiento entre Particiones

El resultado de `StratifiedGroupKFold` ($k = 7$) confirma que los **1.518 pacientes únicos** del catálogo depurado quedaron distribuidos en tres subconjuntos disjuntos (`1.084` en entrenamiento, `217` en validación y `217` en prueba), sin identificadores clínicos compartidos entre particiones. Asimismo, las proporciones de cada categoría se mantienen estables en los tres subconjuntos: **`43.6% - 43.8%` para `Sin hallazgos (0)`**, **`23.3% - 23.4%` para `Cardiomegalia (1)`** y **`32.9% - 33.0%` para `Derrame Pleural (2)`** (`109`, `58` y `82` placas respectivamente tanto en validación como en prueba). Esta correspondencia permite ajustar la temperatura de calibración y los umbrales operativos sobre validación bajo la misma distribución marginal de clases que presenta el conjunto de prueba.


## Parte 04: Ajuste Fino de `DenseNet-121`, Regularización y Entrenamiento a 40 Épocas

La arquitectura **DenseNet-121** (Huang et al., 2017) organiza sus capas en cuatro bloques densos en los que cada unidad convolucional recibe concatenadas las activaciones de todas las capas precedentes del mismo bloque. En el ensayo preliminar del proyecto se congelaron todos los pesos de `model.features` preentrenados en *ImageNet* y se entrenó únicamente la capa lineal de salida (`1024 -> 3`), obteniendo un `52.17%` de exactitud global. Como señala la revisión de Litjens et al. (2017), mientras las primeras capas convolucionales extraen bordes y contrastes generales transferibles entre dominios visuales, adaptar las capas profundas resulta necesario para capturar patrones morfológicos específicos de la imagen médica.

Por ello, esta configuración mantiene congelados los bloques iniciales (`conv0`, `denseblock1` y `denseblock2`) y habilita el ajuste de gradientes en **`denseblock3`, `transition3`, `denseblock4` y `norm5`** en PyTorch (Paszke et al., 2019). El optimizador **AdamW** actualiza esos bloques convolucionales con una tasa de aprendizaje reducida (`lr = 1e-4`) para adaptar los filtros preentrenados de forma gradual, mientras que el cabezal clasificador opera con `lr = 1e-3`. En el preprocesamiento de entrenamiento se prescinde de la inversión horizontal aleatoria (`RandomHorizontalFlip`) —dado que espejar el tórax invierte la posición izquierda habitual de la silueta cardíaca— y se utilizan rotaciones y traslaciones leves (`RandomAffine` de $\pm 7^\circ$) junto con variaciones moderadas de brillo y contraste (`ColorJitter`).

El entrenamiento se ejecuta a lo largo de las **40 épocas completas** sin corte prematuro por *Early Stopping*, con el fin de registrar la evolución completa de la pérdida y del F1-Macro en entrenamiento y validación. Para limitar el sobreajuste se incorporan tres mecanismos de regularización: una capa `Dropout(p=0.35)` antes de la proyección lineal, decaimiento de pesos $L_2$ (`weight_decay = 1e-2`) en `AdamW` y suavizado de etiquetas (`label_smoothing = 0.05`) sobre la pérdida de entropía cruzada ponderada por clase. En cada época donde mejora el F1-Macro de validación se actualiza el archivo de pesos `best_multiclase_model.pth`.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

MEDIA_IMAGENET = [0.485, 0.456, 0.406]
STD_IMAGENET = [0.229, 0.224, 0.225]
DIR_CHECKPOINTS = Path(
    '/content/drive/MyDrive/Unsta/4to año/Proyecto final - Bioinformatica - Procesamiento de imagenes/Dataset/checkpoints'
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Se omite RandomHorizontalFlip porque invertir el hemitórax sitúa la silueta cardíaca
# a la derecha y distorsiona la referencia espacial de Cardiomegalia.
transformacion_entrenamiento = transforms.Compose([
    transforms.RandomAffine(degrees=7, translate=(0.04, 0.04), scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEDIA_IMAGENET, std=STD_IMAGENET),
])

transformacion_evaluacion = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=MEDIA_IMAGENET, std=STD_IMAGENET),
])


class RadiografiasToraxDataset(Dataset):
    def __init__(self, tabla: pd.DataFrame, transform=None):
        self.rutas = tabla['full_path'].tolist()
        self.etiquetas = tabla['Label'].astype(int).tolist()
        self.transform = transform

    def __len__(self) -> int:
        return len(self.etiquetas)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        with Image.open(self.rutas[idx]) as img:
            tensor_img = img.convert('RGB')
        if self.transform is not None:
            tensor_img = self.transform(tensor_img)
        return tensor_img, torch.tensor(self.etiquetas[idx], dtype=torch.long)


def construir_densenet_torax(n_clases: int = 3, dropout: float = 0.35) -> nn.Module:
    """Inicializa DenseNet-121 habilitando gradientes solo desde denseblock3 en adelante."""
    red = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
    for param in red.features.parameters():
        param.requires_grad = False

    bloques_entrenables = [
        red.features.denseblock3,
        red.features.transition3,
        red.features.denseblock4,
        red.features.norm5,
    ]
    for bloque in bloques_entrenables:
        for param in bloque.parameters():
            param.requires_grad = True

    dim_entrada = red.classifier.in_features
    red.classifier = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(dim_entrada, n_clases))
    return red


def ejecutar_epoca_entrenamiento(
    red: nn.Module, loader: DataLoader, criterio: nn.Module, optimizador: optim.Optimizer, disp: torch.device
) -> tuple[float, float]:
    red.train()
    perdida_acumulada = 0.0
    aciertos = 0
    total = 0

    for lotes_img, lotes_y in loader:
        lotes_img, lotes_y = lotes_img.to(disp), lotes_y.to(disp)
        optimizador.zero_grad()
        logits = red(lotes_img)
        perdida = criterio(logits, lotes_y)
        perdida.backward()
        optimizador.step()

        n_lote = lotes_img.size(0)
        perdida_acumulada += perdida.item() * n_lote
        aciertos += (logits.argmax(dim=1) == lotes_y).sum().item()
        total += n_lote

    return perdida_acumulada / total, aciertos / total


@torch.inference_mode()
def evaluar_epoca(
    red: nn.Module, loader: DataLoader, criterio: nn.Module, disp: torch.device
) -> dict[str, object]:
    red.eval()
    perdida_acumulada = 0.0
    probabilidades, objetivos = [], []

    for lotes_img, lotes_y in loader:
        lotes_img, lotes_y = lotes_img.to(disp), lotes_y.to(disp)
        logits = red(lotes_img)
        perdida_acumulada += criterio(logits, lotes_y).item() * lotes_img.size(0)
        probabilidades.append(torch.softmax(logits, dim=1).cpu().numpy())
        objetivos.append(lotes_y.cpu().numpy())

    matriz_probs = np.vstack(probabilidades)
    vector_y = np.concatenate(objetivos)
    predicciones = matriz_probs.argmax(axis=1)

    recalls_por_clase = [
        float((predicciones[vector_y == c] == c).mean()) if np.any(vector_y == c) else 0.0
        for c in (0, 1, 2)
    ]
    return {
        'loss': perdida_acumulada / len(vector_y),
        'acc': float((predicciones == vector_y).mean()),
        'f1_macro': float(f1_score(vector_y, predicciones, average='macro', zero_division=0)),
        'auc_ovr': float(roc_auc_score(vector_y, matriz_probs, multi_class='ovr', average='macro')),
        'recalls': recalls_por_clase,
    }


def graficar_curvas_entrenamiento(historial: dict[str, list], mejor_epoca: int) -> None:
    epocas = range(1, len(historial['train_loss']) + 1)
    recalls = np.asarray(historial['val_recalls'])

    fig, (ax_loss, ax_f1, ax_rec) = plt.subplots(1, 3, figsize=(18, 4.8))

    ax_loss.plot(epocas, historial['train_loss'], label='Entrenamiento', color='#1f77b4', lw=2)
    ax_loss.plot(epocas, historial['val_loss'], label='Validación', color='#ff7f0e', lw=2)
    ax_loss.axvline(mejor_epoca, color='red', ls='--', alpha=0.7, label=f'Mejor época ({mejor_epoca})')
    ax_loss.set(title='Pérdida Ponderada (Cross-Entropy)', xlabel='Época', ylabel='Loss')
    ax_loss.legend()
    ax_loss.grid(True, ls='--', alpha=0.5)

    ax_f1.plot(epocas, historial['val_f1'], label='F1-Macro', color='#2ca02c', lw=2)
    ax_f1.plot(epocas, historial['val_auc'], label='ROC-AUC (OvR)', color='#9467bd', lw=2)
    ax_f1.axvline(mejor_epoca, color='red', ls='--', alpha=0.7)
    ax_f1.set(title='Desempeño en Validación', xlabel='Época', ylabel='Puntaje')
    ax_f1.legend()
    ax_f1.grid(True, ls='--', alpha=0.5)

    for idx, (etiqueta, col) in enumerate([
        ('Sin hallazgos (0)', '#2ca02c'),
        ('Cardiomegalia (1)', '#1f77b4'),
        ('Derrame (2)', '#d62728'),
    ]):
        ax_rec.plot(epocas, recalls[:, idx], label=f'Recall {etiqueta}', color=col, lw=1.8)
    ax_rec.axvline(mejor_epoca, color='red', ls='--', alpha=0.7)
    ax_rec.set(title='Sensibilidad por Categoría (Validación)', xlabel='Época', ylabel='Recall')
    ax_rec.legend()
    ax_rec.grid(True, ls='--', alpha=0.5)

    plt.tight_layout()
    plt.show()


train_loader = DataLoader(
    RadiografiasToraxDataset(df_train, transformacion_entrenamiento),
    batch_size=32, shuffle=True, num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    RadiografiasToraxDataset(df_val, transformacion_evaluacion),
    batch_size=32, shuffle=False, num_workers=2, pin_memory=True,
)
test_loader = DataLoader(
    RadiografiasToraxDataset(df_test, transformacion_evaluacion),
    batch_size=32, shuffle=False, num_workers=2, pin_memory=True,
)

model = construir_densenet_torax(n_clases=3, dropout=0.35).to(device)

pesos_balanceo = compute_class_weight(
    class_weight='balanced', classes=np.array([0, 1, 2]), y=df_train['Label'].to_numpy()
)
criterio_entrenamiento = nn.CrossEntropyLoss(
    weight=torch.tensor(pesos_balanceo, dtype=torch.float32, device=device),
    label_smoothing=0.05,
)

parametros_convolucionales = [p for n, p in model.named_parameters() if p.requires_grad and 'classifier' not in n]
optimizador = optim.AdamW(
    [
        {'params': parametros_convolucionales, 'lr': 1e-4},
        {'params': model.classifier.parameters(), 'lr': 1e-3},
    ],
    weight_decay=1e-2,
)

num_epocas = 40
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizador, T_max=num_epocas, eta_min=1e-6)

DIR_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
ruta_best_model = str(DIR_CHECKPOINTS / 'best_multiclase_model.pth')
ruta_last_model = str(DIR_CHECKPOINTS / 'last_multiclase_model.pth')

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_f1': [], 'val_auc': [], 'val_recalls': []}
mejor_f1_val = -1.0
mejor_epoca = 0

for epoca in range(1, num_epocas + 1):
    t_inicio = time.time()
    loss_tr, acc_tr = ejecutar_epoca_entrenamiento(model, train_loader, criterio_entrenamiento, optimizador, device)
    metricas_val = evaluar_epoca(model, val_loader, criterio_entrenamiento, device)
    scheduler.step()

    history['train_loss'].append(loss_tr)
    history['train_acc'].append(acc_tr)
    history['val_loss'].append(metricas_val['loss'])
    history['val_acc'].append(metricas_val['acc'])
    history['val_f1'].append(metricas_val['f1_macro'])
    history['val_auc'].append(metricas_val['auc_ovr'])
    history['val_recalls'].append(metricas_val['recalls'])

    marca = ''
    if metricas_val['f1_macro'] > mejor_f1_val:
        mejor_f1_val = metricas_val['f1_macro']
        mejor_epoca = epoca
        torch.save(model.state_dict(), ruta_best_model)
        marca = ' [mejor F1]'

    r0, r1, r2 = metricas_val['recalls']
    print(
        f"Época {epoca:02d}/{num_epocas:02d} ({time.time() - t_inicio:.1f}s) | "
        f"Loss Tr/Val: {loss_tr:.4f}/{metricas_val['loss']:.4f} | "
        f"F1 Val: {metricas_val['f1_macro']:.4f} | AUC: {metricas_val['auc_ovr']:.4f} | "
        f"Rec [0:{r0:.2f} 1:{r1:.2f} 2:{r2:.2f}]{marca}"
    )

torch.save(model.state_dict(), ruta_last_model)
graficar_curvas_entrenamiento(history, mejor_epoca)

### Dinámica de Convergencia y Efecto de la Regularización

El registro de las 40 épocas permite evaluar el impacto del ajuste fino parcial. Al habilitar la actualización de `denseblock3` y `denseblock4` con una tasa de aprendizaje diferenciada (`1e-4` en los bloques convolucionales frente a `1e-3` en la capa lineal), el modelo supera el estancamiento del `52.17%` que presentaba la red con pesos congelados. A su vez, el uso conjunto de `Dropout(p=0.35)`, decaimiento de pesos $L_2$ (`1e-2`), *label smoothing* (`0.05`) y el esquema `CosineAnnealingLR` amortigua el sobreajuste en las últimas épocas, resguardando en `best_multiclase_model.pth` los pesos correspondientes al máximo F1-Macro de validación para la evaluación final.

## Parte 05: Evaluación en el Conjunto de Prueba y Análisis en Dos Etapas

Para evaluar los pesos guardados (`best_multiclase_model.pth`) sobre el conjunto de prueba (`249` imágenes de `217` pacientes no vistos durante el entrenamiento ni la validación), el bloque ajusta primero un parámetro de temperatura $T > 0$ minimizando la pérdida logarítmica negativa sobre los *logits* de validación ($\hat{p}_i = \text{Softmax}(z_i / T)$). Con esas probabilidades calibradas, el código reporta en primer lugar el **desempeño multiclase directo** asignando cada placa a la categoría de mayor probabilidad ($\arg\max$).

A continuación, se analiza el comportamiento del clasificador desglosado en dos etapas:
- **Etapa 1 — Triaje binario (`Sin hallazgos` vs. `Patología detectada`):** la probabilidad asignada a la presencia de alguna de las dos patologías se obtiene como $P(\text{Patología}) = P(1) + P(2) = 1 - P(0)$. En el conjunto de prueba existen `140` placas con patología (`58` de `Cardiomegalia` y `82` de `Derrame Pleural`) y `109` placas `Sin hallazgos`. Dado que optimizar el puntaje F1 binario sin contemplar los verdaderos negativos ($TN$) reduce severamente la especificidad cuando la clase positiva es mayoritaria, se calculan sobre la curva ROC del **conjunto de validación** dos puntos de corte adicionales al estándar (`0.50`): el **umbral de Youden** ($\max(\text{Sensibilidad} + \text{Especificidad} - 1)$) y un **umbral de tamizaje** que exige alcanzar al menos un $90\%$ de sensibilidad en validación (`corte = 0.353`).
- **Etapa 2 — Diferenciación condicional entre `Cardiomegalia` y `Derrame Pleural`:** se evalúa la capacidad del modelo para distinguir entre ambas afecciones utilizando la probabilidad condicional $P(\text{Derrame} \mid \text{Patología}) = \frac{P(2)}{P(1) + P(2)}$.

> **Alcance metodológico de la Etapa 2:** Esta segunda etapa se calcula exclusivamente sobre los **`140` casos realmente patológicos** del conjunto de prueba (`test_labels > 0`). Por lo tanto, sus métricas representan un **desempeño condicional (bajo supuesto de oráculo en la Etapa 1)** que mide cuán bien separa la red `Cardiomegalia` de `Derrame Pleural` cuando la presencia de patología ya está confirmada. Ese porcentaje **no** constituye la exactitud global de la cascada completa sobre las `249` radiografías de prueba, ya que una ejecución secuencial extremo a extremo arrastra los falsos negativos descartados en la Etapa 1 y los falsos positivos (`Sin hallazgos`) derivados hacia la Etapa 2.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
)
from torch.utils.data import DataLoader

RUTA_MEJOR_CHECKPOINT = Path(
    '/content/drive/MyDrive/Unsta/4to año/Proyecto final - Bioinformatica - Procesamiento de imagenes/Dataset/checkpoints/best_multiclase_model.pth'
)
ETIQUETAS_CLASE = ['Sin hallazgos (0)', 'Cardiomegalia (1)', 'Derrame (2)']


@torch.no_grad()
def inferir_logits(red: nn.Module, loader: DataLoader, disp: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    red.eval()
    logits_acum, etiquetas_acum = [], []
    for imagenes, etiquetas in loader:
        logits_acum.append(red(imagenes.to(disp)).cpu())
        etiquetas_acum.append(etiquetas.cpu())
    return torch.cat(logits_acum, dim=0), torch.cat(etiquetas_acum, dim=0)


def ajustar_temperatura_validacion(logits_val: torch.Tensor, y_val: torch.Tensor) -> float:
    """Ajusta el escalar T > 0 por L-BFGS minimizando NLL en validación."""
    logits_estandar = logits_val.detach().clone()
    objetivos_estandar = y_val.detach().clone()
    temperatura = nn.Parameter(torch.ones(1))
    criterio_nll = nn.CrossEntropyLoss()
    optimizador_lbfgs = torch.optim.LBFGS([temperatura], lr=0.01, max_iter=50)

    def paso_evaluacion():
        optimizador_lbfgs.zero_grad()
        # Se acota a 0.05 para impedir pasos degenerados de L-BFGS hacia T <= 0.
        perdida = criterio_nll(logits_estandar / temperatura.clamp(min=0.05), objetivos_estandar)
        perdida.backward()
        return perdida

    optimizador_lbfgs.step(paso_evaluacion)
    return float(temperatura.clamp(min=0.05).item())


def calcular_umbrales_roc_triaje(
    y_val_bin: np.ndarray, probs_anormal_val: np.ndarray, sensibilidad_minima: float = 0.90
) -> tuple[float, float]:
    """Obtiene los cortes de Youden y de tamizaje (Sens >= 90% en validación) sobre la curva ROC."""
    fpr, tpr, umbrales = roc_curve(y_val_bin, probs_anormal_val)
    especificidad = 1.0 - fpr

    corte_youden = float(umbrales[np.argmax(tpr + especificidad - 1.0)])
    candidatos_clinicos = np.where(tpr >= sensibilidad_minima)[0]
    if len(candidatos_clinicos) == 0:
        return corte_youden, corte_youden

    mejor_idx_clinico = candidatos_clinicos[np.argmax(especificidad[candidatos_clinicos])]
    return corte_youden, float(umbrales[mejor_idx_clinico])


def evaluar_regimen_triaje(
    y_bin: np.ndarray, probs_anormal: np.ndarray, umbral: float, nombre_regimen: str
) -> np.ndarray:
    pred_bin = (probs_anormal >= umbral).astype(int)
    exactitud = accuracy_score(y_bin, pred_bin)
    vpp, sensibilidad, f1, _ = precision_recall_fscore_support(
        y_bin, pred_bin, average='binary', zero_division=0
    )
    matriz = confusion_matrix(y_bin, pred_bin)
    tn, fp, fn, tp = matriz.ravel()
    especificidad = tn / (tn + fp) if (tn + fp) else 0.0
    auc_bin = roc_auc_score(y_bin, probs_anormal)

    print(f'\n[{nombre_regimen} | corte = {umbral:.2f}]')
    print(f'  Accuracy:      {exactitud:.4f} | F1: {f1:.4f} | ROC-AUC: {auc_bin:.4f}')
    print(f'  Sensibilidad:  {sensibilidad:.4f} ({tp}/{tp + fn} con patología) | VPP: {vpp:.4f}')
    print(f'  Especificidad: {especificidad:.4f} ({tn}/{tn + fp} sin hallazgos)')
    return matriz


if isinstance(model.classifier, nn.Linear):
    dim_in = model.classifier.in_features
    model.classifier = nn.Sequential(nn.Dropout(p=0.35), nn.Linear(dim_in, 3)).to(device)

model.load_state_dict(torch.load(RUTA_MEJOR_CHECKPOINT, map_location=device))
model.to(device)

val_logits_t, val_labels_t = inferir_logits(model, val_loader, device)
test_logits_t, test_labels_t = inferir_logits(model, test_loader, device)

T_opt = ajustar_temperatura_validacion(val_logits_t, val_labels_t)
print(f'Temperatura de calibración (Validación): T = {T_opt:.3f}')

val_probs = torch.softmax(val_logits_t / T_opt, dim=1).numpy()
val_labels = val_labels_t.numpy()
test_probs = torch.softmax(test_logits_t / T_opt, dim=1).numpy()
test_labels = test_labels_t.numpy()

preds_argmax = test_probs.argmax(axis=1)
acc_multi = accuracy_score(test_labels, preds_argmax)
auc_multi_ovr = roc_auc_score(test_labels, test_probs, multi_class='ovr', average='macro')

print(f'\n Rendimiento Multiclase Global (Test, n={len(test_labels)}) ')
print(f'Accuracy: {acc_multi:.4f} | ROC-AUC Macro (OvR): {auc_multi_ovr:.4f}')
print(classification_report(test_labels, preds_argmax, target_names=ETIQUETAS_CLASE, zero_division=0))

cm_multi = confusion_matrix(test_labels, preds_argmax)
fig, (ax_cm, ax_pr) = plt.subplots(1, 2, figsize=(13, 4.8))
sns.heatmap(
    cm_multi, annot=True, fmt='d', cmap='Blues', ax=ax_cm,
    xticklabels=['Sin hallazgos', 'Cardio', 'Derrame'],
    yticklabels=['Sin hallazgos', 'Cardio', 'Derrame'],
)
ax_cm.set(title='Matriz de Confusión Multiclase (Argmax)', ylabel='Categoría Real', xlabel='Predicción')

for idx, (nombre, col) in enumerate(zip(ETIQUETAS_CLASE, ['#2ca02c', '#1f77b4', '#d62728'])):
    binario_clase = (test_labels == idx).astype(int)
    prec_c, rec_c, _ = precision_recall_curve(binario_clase, test_probs[:, idx])
    ap_c = average_precision_score(binario_clase, test_probs[:, idx])
    ax_pr.plot(rec_c, prec_c, color=col, lw=2, label=f'{nombre} (AP={ap_c:.2f})')
ax_pr.set(title='Curvas Precision-Recall por Categoría', xlabel='Sensibilidad', ylabel='Precisión')
ax_pr.legend()
ax_pr.grid(True, ls='--', alpha=0.5)
plt.tight_layout()
plt.show()

val_labels_bin = (val_labels > 0).astype(int)
val_probs_anormal = val_probs[:, 1] + val_probs[:, 2]
umbral_youden, umbral_clinico = calcular_umbrales_roc_triaje(val_labels_bin, val_probs_anormal)
print(f'Umbrales definidos en Validación -> Youden: {umbral_youden:.3f} | Tamizaje (Sens Val>=90%): {umbral_clinico:.3f}')

test_labels_bin = (test_labels > 0).astype(int)
test_probs_anormal = test_probs[:, 1] + test_probs[:, 2]

cm_std = evaluar_regimen_triaje(test_labels_bin, test_probs_anormal, 0.50, 'Etapa 1 - Estándar')
cm_youden = evaluar_regimen_triaje(test_labels_bin, test_probs_anormal, umbral_youden, 'Etapa 1 - Youden')
cm_clin = evaluar_regimen_triaje(
    test_labels_bin, test_probs_anormal, umbral_clinico, 'Etapa 1 - Tamizaje (Sens Val>=90%)'
)

fig, (ax_y, ax_c, ax_h) = plt.subplots(1, 3, figsize=(16, 4.4))
sns.heatmap(
    cm_youden, annot=True, fmt='d', cmap='Greens', ax=ax_y,
    xticklabels=['Sin hallazgos (0)', 'Patología (1+2)'],
    yticklabels=['Sin hallazgos (0)', 'Patología (1+2)'],
)
ax_y.set(title=f'Triaje Youden (>= {umbral_youden:.2f})', ylabel='Real', xlabel='Predicción')

sns.heatmap(
    cm_clin, annot=True, fmt='d', cmap='Blues', ax=ax_c,
    xticklabels=['Sin hallazgos (0)', 'Patología (1+2)'],
    yticklabels=['Sin hallazgos (0)', 'Patología (1+2)'],
)
ax_c.set(title=f'Triaje Tamizaje (>= {umbral_clinico:.2f})', ylabel='Real', xlabel='Predicción')

ax_h.hist(
    test_probs_anormal[test_labels_bin == 0], bins=20, alpha=0.6,
    color='green', label='Sin hallazgos (0)', density=True,
)
ax_h.hist(
    test_probs_anormal[test_labels_bin == 1], bins=20, alpha=0.6,
    color='red', label='Patología (1+2)', density=True,
)
ax_h.axvline(umbral_youden, color='darkgreen', ls='--', lw=2, label=f'Youden ({umbral_youden:.2f})')
ax_h.axvline(umbral_clinico, color='navy', ls=':', lw=2, label=f'Tamizaje ({umbral_clinico:.2f})')
ax_h.set(title='Densidad de P(Patología)', xlabel='P(Cardio) + P(Derrame)', ylabel='Densidad')
ax_h.legend()
plt.tight_layout()
plt.show()

mask_enfermos = test_labels > 0
labels_etapa2 = (test_labels[mask_enfermos] == 2).astype(int)
probs_cardio = test_probs[mask_enfermos, 1]
probs_derrame = test_probs[mask_enfermos, 2]
probs_cond_derrame = probs_derrame / np.clip(probs_cardio + probs_derrame, 1e-8, None)
preds_etapa2 = (probs_cond_derrame >= 0.50).astype(int)

auc_etapa2 = roc_auc_score(labels_etapa2, probs_cond_derrame)
print(
    f'\n[Etapa 2 Condicional (solo n={int(mask_enfermos.sum())} casos patológicos reales) '
    f'- Cardiomegalia vs. Derrame | ROC-AUC = {auc_etapa2:.4f}]'
)
print(classification_report(
    labels_etapa2, preds_etapa2, target_names=['Cardiomegalia (1)', 'Derrame Pleural (2)'], zero_division=0
))

Temperatura de calibración (Validación): T = 1.121

 Rendimiento Multiclase Global (Test, n=249) 
Accuracy: 0.6908 | ROC-AUC Macro (OvR): 0.8654
                   precision    recall  f1-score   support

Sin hallazgos (0)       0.69      0.72      0.70       109
Cardiomegalia (1)       0.81      0.50      0.62        58
      Derrame (2)       0.65      0.79      0.71        82

         accuracy                           0.69       249
        macro avg       0.72      0.67      0.68       249
     weighted avg       0.70      0.69      0.69       249

Umbrales definidos en Validación -> Youden: 0.578 | Tamizaje (Sens Val>=90%): 0.353

[Etapa 1 - Estándar | corte = 0.50]
  Accuracy:      0.7430 | F1: 0.7714 | ROC-AUC: 0.8231
  Sensibilidad:  0.7714 (108/140 con patología) | VPP: 0.7714
  Especificidad: 0.7064 (77/109 sin hallazgos)

[Etapa 1 - Youden | corte = 0.58]
  Accuracy:      0.7390 | F1: 0.7636 | ROC-AUC: 0.8231
  Sensibilidad:  0.7500 (105/140 con patología) | VPP: 0.7778
  

### Visualización de Resultados y Exportación de Figuras

El siguiente bloque define cinco funciones que grafican los resultados sobre el conjunto de prueba (`n = 249`) y exportan cada figura en resolución de `300 DPI` dentro del directorio `figuras_presentacion/`:

1. **`graficar_comparativa_evolucion_modelo()`**: compara el ensayo inicial (red congelada y catálogo sin depurar) frente al modelo final con ajuste fino y curación de etiquetas, tanto en métricas multiclase como en el equilibrio entre sensibilidad y especificidad de la Etapa 1.
2. **`graficar_analisis_multiclase_completo()`**: presenta la matriz de confusión normalizada por fila (porcentaje de acierto sobre el total real de cada categoría), las curvas ROC *One-vs-Rest* de las tres clases y el desglose de precisión, sensibilidad y F1.
3. **`graficar_triaje_etapa1_detallado()`**: grafica la curva ROC del triaje binario (`ROC-AUC = 0.8231`) indicando los tres umbrales evaluados (`0.35`, `0.50` y `0.58`), la variación continua de sensibilidad, especificidad e índice J de Youden según el umbral, y el recuento de verdaderos y falsos positivos/negativos (`TP`, `TN`, `FP`, `FN`).
4. **`graficar_etapa2_diferencial()`**: muestra la matriz de confusión condicional entre `Cardiomegalia` y `Derrame Pleural` evaluada sobre los `140` casos patológicos reales (`ROC-AUC = 0.9521`), su curva ROC y Precisión-Sensibilidad, y el histograma de la probabilidad condicional $P(\text{Derrame} \mid \text{Patología})$.
5. **`graficar_galeria_predicciones_clinicas()`**: selecciona placas del conjunto de prueba y muestra sobre cada radiografía la categoría real junto con las probabilidades calculadas en la Etapa 1 y en la Etapa 2.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from sklearn.metrics import (
    auc,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    precision_recall_fscore_support,
    roc_curve,
)

RUTA_DATASET_DRIVE = Path(
    '/content/drive/MyDrive/Unsta/4to año/Proyecto final - Bioinformatica - Procesamiento de imagenes/Dataset'
)
DIR_FIGURAS = (RUTA_DATASET_DRIVE / 'figuras_presentacion') if RUTA_DATASET_DRIVE.exists() else Path('./figuras_presentacion')
DIR_FIGURAS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'font.size': 10.5,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.labelsize': 10.5,
})


def _anotar_barras(ax: plt.Axes, barras, formato: str = '{:.2f}', offset: float = 0.015, fontsize: int = 9) -> None:
    for barra in barras:
        altura = barra.get_height()
        ax.text(
            barra.get_x() + barra.get_width() / 2.0,
            altura + offset,
            formato.format(altura),
            ha='center', va='bottom', fontsize=fontsize, fontweight='bold', color='#1e293b',
        )


def _matriz_con_porcentajes(y_real: np.ndarray, y_pred: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    cm = confusion_matrix(y_real, y_pred)
    pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100.0
    etiquetas = np.array([
        [f'{conteo}\n({porcentaje:.1f}%)' for conteo, porcentaje in zip(fila_c, fila_p)]
        for fila_c, fila_p in zip(cm, pct)
    ])
    return pct, etiquetas


def graficar_comparativa_evolucion_modelo(dir_salida: Path = DIR_FIGURAS) -> Path:
    metricas = ['Accuracy\nGlobal', 'F1-Score\nMacro', 'F1\nSin hallazgos (0)', 'F1\nCardio (1)', 'F1\nDerrame (2)']
    modelo_inicial = [0.5217, 0.5100, 0.4700, 0.4700, 0.5900]
    modelo_final = [0.6908, 0.6800, 0.7000, 0.6200, 0.7100]

    cortes = ['Inicial\n(0.50)', 'Inicial\n(0.57)', 'Tamizaje\n(0.35)', 'Estándar\n(0.50)', 'Youden\n(0.58)']
    sens = [90.70, 81.40, 83.57, 77.14, 75.00]
    esp = [16.00, 26.40, 64.22, 70.64, 72.48]
    acc = [57.80, 57.10, 75.10, 74.30, 73.90]

    fig, (ax_multi, ax_triaje) = plt.subplots(1, 2, figsize=(15.5, 5.4))

    pos = np.arange(len(metricas))
    ancho = 0.36
    b_ini = ax_multi.bar(pos - ancho / 2, modelo_inicial, ancho, label='Inicial (Red congelada / CSV sin depurar)', color='#94a3b8', edgecolor='#334155')
    b_fin = ax_multi.bar(pos + ancho / 2, modelo_final, ancho, label='Final (Fine-Tuning / CSV depurado)', color='#0284c7', edgecolor='#0f172a')
    _anotar_barras(ax_multi, b_ini)
    _anotar_barras(ax_multi, b_fin)
    ax_multi.set(ylim=(0, 0.88), ylabel='Puntaje (0.0 - 1.0)', title='a) Desempeño Multiclase: Inicial vs. Final', xticks=pos, xticklabels=metricas)
    ax_multi.legend(loc='upper left')
    ax_multi.grid(axis='y', ls='--', alpha=0.5)

    pos_t = np.arange(len(cortes))
    ancho_t = 0.26
    b_s = ax_triaje.bar(pos_t - ancho_t, sens, ancho_t, label='Sensibilidad (%)', color='#dc2626')
    b_e = ax_triaje.bar(pos_t, esp, ancho_t, label='Especificidad (%)', color='#16a34a')
    b_a = ax_triaje.bar(pos_t + ancho_t, acc, ancho_t, label='Accuracy Binario (%)', color='#2563eb')
    for grupo in (b_s, b_e, b_a):
        _anotar_barras(ax_triaje, grupo, formato='{:.1f}%', offset=1.2, fontsize=8)

    ax_triaje.axvline(1.5, color='#334155', ls='--', lw=1.5)
    ax_triaje.set(ylim=(0, 108), ylabel='Porcentaje (%)', title='b) Etapa 1 (Triaje): Recuperación de la Especificidad', xticks=pos_t, xticklabels=cortes)
    ax_triaje.legend(loc='upper right')
    ax_triaje.grid(axis='y', ls='--', alpha=0.5)

    plt.tight_layout()
    ruta_png = dir_salida / '01_comparativa_evolucion_modelo.png'
    plt.savefig(ruta_png, bbox_inches='tight')
    plt.show()
    return ruta_png


def graficar_analisis_multiclase_completo(
    y_real: np.ndarray, probs: np.ndarray, dir_salida: Path = DIR_FIGURAS
) -> Path:
    clases = ['Sin hallazgos (0)', 'Cardiomegalia (1)', 'Derrame Pleural (2)']
    colores = ['#16a34a', '#0284c7', '#dc2626']
    y_pred = probs.argmax(axis=1)
    pct_fila, anotaciones = _matriz_con_porcentajes(y_real, y_pred)

    fig, (ax_cm, ax_roc, ax_met) = plt.subplots(1, 3, figsize=(18, 5.2))

    sns.heatmap(
        pct_fila, annot=anotaciones, fmt='', cmap='Blues', vmin=0, vmax=100,
        xticklabels=['Sin hallazgos (0)', 'Cardio (1)', 'Derrame (2)'],
        yticklabels=clases, annot_kws={'size': 10, 'weight': 'bold'}, ax=ax_cm,
    )
    ax_cm.set(title='a) Matriz Multiclase (% por categoría real)', ylabel='Categoría Real', xlabel='Predicción (Argmax)')

    aucs = []
    for idx, (nombre, col) in enumerate(zip(clases, colores)):
        binario = (y_real == idx).astype(int)
        fpr, tpr, _ = roc_curve(binario, probs[:, idx])
        valor_auc = auc(fpr, tpr)
        aucs.append(valor_auc)
        ax_roc.plot(fpr, tpr, color=col, lw=2.3, label=f'{nombre} (AUC = {valor_auc:.3f})')

    ax_roc.plot([0, 1], [0, 1], color='gray', ls='--', lw=1.2, label=f'Macro AUC = {np.mean(aucs):.4f}')
    ax_roc.set(xlim=(-0.02, 1.02), ylim=(-0.02, 1.05), xlabel='1 - Especificidad (FPR)', ylabel='Sensibilidad (TPR)', title='b) Curvas ROC One-vs-Rest (Test)')
    ax_roc.legend(loc='lower right')
    ax_roc.grid(True, ls='--', alpha=0.5)

    prec, rec, f1, _ = precision_recall_fscore_support(y_real, y_pred, labels=[0, 1, 2], zero_division=0)
    pos = np.arange(3)
    ancho = 0.25
    for despl, serie, etiqueta, col in [
        (-ancho, prec, 'Precisión', '#0ea5e9'),
        (0.0, rec, 'Recall (Sens.)', '#f59e0b'),
        (ancho, f1, 'F1-Score', '#10b981'),
    ]:
        barras = ax_met.bar(pos + despl, serie, ancho, label=etiqueta, color=col, edgecolor='black')
        _anotar_barras(ax_met, barras, offset=0.015, fontsize=8)

    ax_met.set(ylim=(0, 1.05), ylabel='Puntaje', title='c) Desglose por Categoría (Test)', xticks=pos, xticklabels=['Sin hallazgos (0)', 'Cardio (1)', 'Derrame (2)'])
    ax_met.legend(loc='upper right')
    ax_met.grid(axis='y', ls='--', alpha=0.5)

    plt.tight_layout()
    ruta_png = dir_salida / '02_analisis_multiclase_completo.png'
    plt.savefig(ruta_png, bbox_inches='tight')
    plt.show()
    return ruta_png


def graficar_triaje_etapa1_detallado(
    y_bin: np.ndarray, probs_anormal: np.ndarray, u_youden: float, u_clinico: float, dir_salida: Path = DIR_FIGURAS
) -> Path:
    fpr, tpr, _ = roc_curve(y_bin, probs_anormal)
    fig, (ax_roc, ax_curvas, ax_conteo) = plt.subplots(1, 3, figsize=(18, 5.2))

    ax_roc.plot(fpr, tpr, color='#0f172a', lw=2.5, label=f'ROC Triaje (AUC = {auc(fpr, tpr):.4f})')
    ax_roc.plot([0, 1], [0, 1], color='gray', ls='--')

    regimenes = [
        ('Tamizaje (Val>=90%)', u_clinico, '#2563eb', 'o'),
        ('Estándar (0.50)', 0.50, '#d97706', 's'),
        ('Youden', u_youden, '#16a34a', '^'),
    ]
    for nombre, corte, col, marcador in regimenes:
        pred = (probs_anormal >= corte).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_bin, pred).ravel()
        sens_c, esp_c = tp / (tp + fn), tn / (tn + fp)
        ax_roc.scatter(1 - esp_c, sens_c, color=col, s=100, marker=marcador, zorder=5, edgecolors='black', label=f'{nombre} ({corte:.2f}): Sens={sens_c*100:.1f}%, Esp={esp_c*100:.1f}%')

    ax_roc.set(xlim=(-0.02, 1.02), ylim=(-0.02, 1.05), xlabel='1 - Especificidad (Falsos Positivos)', ylabel='Sensibilidad (Verdaderos Positivos)', title='a) Curva ROC de Etapa 1 y Puntos de Corte')
    ax_roc.legend(loc='lower right', fontsize=8.5)
    ax_roc.grid(True, ls='--', alpha=0.5)

    malla = np.linspace(0.05, 0.95, 180)
    sens_m, esp_m, j_m = [], [], []
    for c in malla:
        tn, fp, fn, tp = confusion_matrix(y_bin, (probs_anormal >= c).astype(int)).ravel()
        s, e = tp / (tp + fn), tn / (tn + fp)
        sens_m.append(s)
        esp_m.append(e)
        j_m.append(s + e - 1.0)

    ax_curvas.plot(malla, sens_m, color='#dc2626', lw=2.2, label='Sensibilidad')
    ax_curvas.plot(malla, esp_m, color='#16a34a', lw=2.2, label='Especificidad')
    ax_curvas.plot(malla, j_m, color='#7c3aed', ls='-.', lw=1.8, label='Índice J de Youden')
    ax_curvas.axvline(u_clinico, color='#2563eb', ls=':', lw=2, label=f'Tamizaje ({u_clinico:.2f})')
    ax_curvas.axvline(0.50, color='#d97706', ls='--', lw=1.8, label='Estándar (0.50)')
    ax_curvas.axvline(u_youden, color='#16a34a', ls='--', lw=2, label=f'Youden ({u_youden:.2f})')
    ax_curvas.set(xlim=(0.05, 0.95), ylim=(0.0, 1.05), xlabel='Umbral de probabilidad P(Patología)', ylabel='Valor de la métrica', title='b) Equilibrio Sensibilidad vs. Especificidad')
    ax_curvas.legend(loc='center left', fontsize=8.5)
    ax_curvas.grid(True, ls='--', alpha=0.5)

    filas_desglose = []
    for nombre, corte, _, _ in regimenes:
        tn, fp, fn, tp = confusion_matrix(y_bin, (probs_anormal >= corte).astype(int)).ravel()
        filas_desglose.append((f'{nombre}\n({corte:.2f})', tp, tn, fp, fn))

    pos_r = np.arange(len(filas_desglose))
    ancho_r = 0.20
    for despl, idx_col, etiq, col in [
        (-1.5 * ancho_r, 1, 'TP (Patología detectada)', '#16a34a'),
        (-0.5 * ancho_r, 2, 'TN (Sin hallazgos desc.)', '#0284c7'),
        (0.5 * ancho_r, 3, 'FP (Falsa alarma)', '#f59e0b'),
        (1.5 * ancho_r, 4, 'FN (Patología omitida)', '#dc2626'),
    ]:
        valores = [f[idx_col] for f in filas_desglose]
        barras = ax_conteo.bar(pos_r + despl, valores, ancho_r, label=etiq, color=col, edgecolor='black')
        _anotar_barras(ax_conteo, barras, formato='{:.0f}', offset=1.5, fontsize=8)

    ax_conteo.set(ylim=(0, 145), ylabel='Número de Pacientes / Placas (n=249)', title='c) Desglose de Casos en Prueba por Umbral', xticks=pos_r, xticklabels=[f[0] for f in filas_desglose])
    ax_conteo.legend(loc='upper right', fontsize=8.5)
    ax_conteo.grid(axis='y', ls='--', alpha=0.5)

    plt.tight_layout()
    ruta_png = dir_salida / '03_triaje_etapa1_detallado.png'
    plt.savefig(ruta_png, bbox_inches='tight')
    plt.show()
    return ruta_png


def graficar_etapa2_diferencial(
    y_etapa2: np.ndarray, probs_cond: np.ndarray, pred_etapa2: np.ndarray, dir_salida: Path = DIR_FIGURAS
) -> Path:
    pct_fila, anotaciones = _matriz_con_porcentajes(y_etapa2, pred_etapa2)
    fig, (ax_cm, ax_curvas, ax_hist) = plt.subplots(1, 3, figsize=(18, 5.1))

    sns.heatmap(
        pct_fila, annot=anotaciones, fmt='', cmap='YlGnBu', vmin=0, vmax=100,
        xticklabels=['Cardiomegalia (1)', 'Derrame Pleural (2)'],
        yticklabels=['Cardiomegalia (1)', 'Derrame Pleural (2)'],
        annot_kws={'size': 11, 'weight': 'bold'}, ax=ax_cm,
    )
    ax_cm.set(title='a) Matriz Condicional Etapa 2 (n = 140 Patológicos)', ylabel='Patología Real', xlabel='Predicción (Corte 0.50)')

    fpr2, tpr2, _ = roc_curve(y_etapa2, probs_cond)
    prec2, rec2, _ = precision_recall_curve(y_etapa2, probs_cond)
    ax_curvas.plot(fpr2, tpr2, color='#0d9488', lw=2.5, label=f'ROC Diferencial (AUC = {auc(fpr2, tpr2):.4f})')
    ax_curvas.plot(rec2, prec2, color='#4f46e5', ls='--', lw=2.2, label=f'Precision-Recall (AP = {average_precision_score(y_etapa2, probs_cond):.4f})')
    ax_curvas.plot([0, 1], [0, 1], color='gray', ls=':')
    ax_curvas.set(xlim=(-0.02, 1.02), ylim=(-0.02, 1.05), xlabel='1 - Especificidad / Recall', ylabel='Sensibilidad / Precisión', title='b) Discriminación Condicional en Etapa 2')
    ax_curvas.legend(loc='lower left')
    ax_curvas.grid(True, ls='--', alpha=0.5)

    ax_hist.hist(probs_cond[y_etapa2 == 0], bins=18, alpha=0.65, color='#0284c7', edgecolor='black', density=True, label='Cardiomegalia (n=58)')
    ax_hist.hist(probs_cond[y_etapa2 == 1], bins=18, alpha=0.65, color='#dc2626', edgecolor='black', density=True, label='Derrame Pleural (n=82)')
    ax_hist.axvline(0.50, color='#0f172a', ls='--', lw=2.2, label='Corte Diferencial (0.50)')
    ax_hist.set(xlim=(0.0, 1.0), xlabel='P(Derrame | Patología)', ylabel='Densidad', title='c) Separación de Probabilidades Condicionales')
    ax_hist.legend(loc='upper center')
    ax_hist.grid(True, ls='--', alpha=0.5)

    plt.tight_layout()
    ruta_png = dir_salida / '04_etapa2_diagnostico_diferencial.png'
    plt.savefig(ruta_png, bbox_inches='tight')
    plt.show()
    return ruta_png


def graficar_galeria_predicciones_clinicas(
    tabla_test: pd.DataFrame | None, y_real: np.ndarray, probs: np.ndarray, u_clinico: float, dir_salida: Path = DIR_FIGURAS
) -> Path | None:
    if tabla_test is None or 'full_path' not in tabla_test.columns:
        return None

    df_vis = tabla_test.reset_index(drop=True)
    nombres = {0: 'Sin hallazgos (0)', 1: 'Cardiomegalia (1)', 2: 'Derrame Pleural (2)'}
    seleccion = []

    pred_argmax = probs.argmax(axis=1)
    for clase in (0, 1, 2):
        aciertos_clase = np.where((y_real == clase) & (pred_argmax == clase))[0]
        fuente = aciertos_clase if len(aciertos_clase) >= 2 else np.where(y_real == clase)[0]
        orden_confianza = fuente[np.argsort(probs[fuente, clase])[::-1]]
        seleccion.extend(orden_confianza[:2])

    fig, axes = plt.subplots(2, 3, figsize=(15, 9.2))
    for ax, idx_fila in zip(axes.ravel(), seleccion[:6]):
        ruta_img = Path(df_vis.loc[idx_fila, 'full_path'])
        clase_real = int(y_real[idx_fila])
        _, p_cardio, p_derrame = probs[idx_fila]
        p_patologia = p_cardio + p_derrame

        if p_patologia < u_clinico:
            diagnostico_sistema = 'Sin hallazgos (Descarte Etapa 1)'
            texto_etapa2 = 'Sin pase a Etapa 2'
        else:
            p_cond = p_derrame / max(p_patologia, 1e-8)
            diagnostico_sistema = 'Derrame Pleural (Etapa 2)' if p_cond >= 0.50 else 'Cardiomegalia (Etapa 2)'
            texto_etapa2 = f'P(Derr|Pat): {p_cond*100:.1f}%'

        if ruta_img.exists():
            with Image.open(ruta_img) as img:
                ax.imshow(img.convert('L'), cmap='bone')
        ax.axis('off')
        color_rotulo = '#15803d' if pred_argmax[idx_fila] == clase_real else '#dc2626'
        ax.set_title(
            f'Real: {nombres[clase_real]}\n{diagnostico_sistema}\nEtapa 1 P(Pat): {p_patologia*100:.1f}% | {texto_etapa2}',
            fontsize=9.5, color=color_rotulo, fontweight='bold',
        )

    plt.suptitle('Inferencia sobre Radiografías del Conjunto de Prueba', fontsize=13.5, fontweight='bold', y=1.01)
    plt.tight_layout()
    ruta_png = dir_salida / '05_galeria_predicciones_clinicas.png'
    plt.savefig(ruta_png, bbox_inches='tight')
    plt.show()
    return ruta_png


graficar_comparativa_evolucion_modelo()
if 'test_labels' in globals() and 'test_probs' in globals():
    graficar_analisis_multiclase_completo(test_labels, test_probs)
    graficar_triaje_etapa1_detallado(test_labels_bin, test_probs_anormal, umbral_youden, umbral_clinico)
    graficar_etapa2_diferencial(labels_etapa2, probs_cond_derrame, preds_etapa2)
    graficar_galeria_predicciones_clinicas(globals().get('df_test'), test_labels, test_probs, umbral_clinico)

### Discusión de los Resultados en el Conjunto de Prueba

En la **evaluación multiclase directa** (`n = 249`), la exactitud global aumentó del `52.17%` obtenido con la red congelada al **`69.08%`** (`+16.91 puntos porcentuales`), con un F1-Macro de **`0.6800`** y un área bajo la curva ROC (*One-vs-Rest*) de **`0.8654`**. El desglose por categoría muestra que:
- **`Sin hallazgos (0)`** ($n = 109$) duplicó su sensibilidad pasando de `0.36` a **`0.72`**, con una precisión de **`0.69`** y un F1 de **`0.70`**.
- **`Derrame Pleural (2)`** ($n = 82$) elevó su precisión de `0.46` a **`0.65`** manteniendo una sensibilidad de **`0.79`** (F1 = **`0.71`**), dejando de absorber de forma sistemática los casos de las demás clases.
- **`Cardiomegalia (1)`** ($n = 58$) obtuvo la precisión más alta del conjunto (**`0.81`** frente al `0.54` inicial), pero conservó una sensibilidad moderada de **`0.50`** (F1 = **`0.62`**).

En la **Etapa 1 (Triaje binario `Sin hallazgos` vs. `Patología detectada`)**, la calibración por temperatura en validación convergió en $T = 1.121$ y el área bajo la curva ROC en prueba alcanzó **`0.8231`** (frente al `0.7254` inicial). La comparación de los tres puntos de corte sobre las `249` placas de prueba (`140` con patología y `109` sin hallazgos) refleja el compromiso entre sensibilidad y especificidad:
- **Umbral de tamizaje (`0.353` $\approx 0.35$):** este umbral se definió exigiendo una sensibilidad mínima del $90\%$ **en el conjunto de validación**. Al evaluarlo sobre pacientes nuevos en el **conjunto de prueba**, alcanza una sensibilidad del **`83.57%`** ($117/140$ casos patológicos detectados), un valor predictivo positivo (VPP) del **`75.00%`**, una especificidad del **`64.22%`** ($70/109$ estudios `Sin hallazgos` descartados, frente al `16.00%` inicial) y la mayor exactitud binaria de la etapa (**`75.10%`**, F1 = **`0.7905`**). La diferencia entre el $90\%$ fijado en validación y el $83.57\%$ medido en prueba responde a la variación muestral entre particiones de pacientes independientes.
- **Umbral estándar (`0.50`):** registra una exactitud del **`74.30%`** (F1 = **`0.7714`**), con una sensibilidad del **`77.14%`** ($108/140$) y una especificidad del **`70.64%`** ($77/109$).
- **Umbral de Youden (`0.578` $\approx 0.58$):** prioriza la especificidad llegando al **`72.48%`** ($79/109$) y a un VPP del **`77.78%`**, a cambio de reducir la sensibilidad en prueba al **`75.00%`** ($105/140$, exactitud = **`73.90%`**).

En la **Etapa 2 condicional (`Cardiomegalia` vs. `Derrame Pleural`)**, evaluada exclusivamente sobre los **`140` casos realmente patológicos** del conjunto de prueba mediante la probabilidad condicional $P(\text{Derrame} \mid \text{Patología})$, el modelo alcanza un **ROC-AUC de `0.9521`** y una exactitud condicional del **`88.00%`** (F1-Macro = **`0.8700`**): `Cardiomegalia (1)` registra una precisión de **`0.92`**, sensibilidad de **`0.78`** y F1 de **`0.84`**, mientras que `Derrame Pleural (2)` obtiene una precisión de **`0.86`**, sensibilidad de **`0.95`** y F1 de **`0.90`**.

Es necesario distinguir esta métrica condicional del rendimiento global del sistema. El **`88.00%` de exactitud corresponde únicamente al subconjunto de `140` placas con patología confirmada** y no representa la exactitud extremo a extremo de toda la cascada sobre los `249` estudios de prueba (la cual en clasificación conjunta de tres vías es del **`69.08%`**). Sin embargo, el contraste entre ambas evaluaciones aporta un dato diagnóstico concreto sobre el comportamiento del modelo: cuando la comparación se restringe a `Cardiomegalia` frente a `Derrame Pleural`, la sensibilidad de `Cardiomegalia` sube de `0.50` a `0.78` y su precisión a `0.92`, lo que demuestra que los falsos negativos de `Cardiomegalia` en el clasificador multiclase provienen mayoritariamente de su confusión con la clase `Sin hallazgos (0)` y no de una confusión morfológica con `Derrame Pleural (2)`.

---
## Conclusiones Finales

| Etapa / Dimensión | Métrica | Modelo Inicial (Red Congelada / CSV sin depurar) | Modelo Final (Fine-Tuning / CSV depurado) | Diferencia |
| :--- | :--- | :---: | :---: | :---: |
| **Multiclase (3 vías, $n=249$)** | Accuracy Global | `52.17%` | **`69.08%`** | `+16.91 pts` |
| **Multiclase (3 vías, $n=249$)** | F1-Score Macro | `0.5100` | **`0.6800`** | `+0.1700` |
| **Multiclase (3 vías, $n=249$)** | ROC-AUC Macro (OvR) | — | **`0.8654`** | — |
| **Por categoría ($n=249$)** | F1 `Sin hallazgos (0)` | `0.47` (Recall `0.36`) | **`0.70` (Recall `0.72`)** | `+0.23` |
| **Por categoría ($n=249$)** | F1 `Cardiomegalia (1)` | `0.47` (Precisión `0.54`) | **`0.62` (Precisión `0.81`)** | `+0.15` |
| **Por categoría ($n=249$)** | F1 `Derrame Pleural (2)` | `0.59` (Precisión `0.46`) | **`0.71` (Precisión `0.65`)** | `+0.12` |
| **Etapa 1: Triaje ($n=249$)** | ROC-AUC Binario | `0.7254` | **`0.8231`** | `+0.0977` |
| **Etapa 1: Triaje ($n=249$)** | Corte Youden (`0.58`) | Esp. `26.40%` (corte `0.57`) | **Esp. `72.48%`** (Sens. `75.00%`, Acc `73.90%`) | `+46.08 pts Esp.` |
| **Etapa 1: Triaje ($n=249$)** | Corte Estándar (`0.50`) | Esp. `16.00%` (corte `0.50`) | **Esp. `70.64%`** (Sens. `77.14%`, Acc `74.30%`) | `+54.64 pts Esp.` |
| **Etapa 1: Triaje ($n=249$)** | Corte Tamizaje (`0.35`, Val $\ge 90\%$) | Esp. `16.00%` (corte `0.50`) | **Esp. `64.22%`** (Sens. Test `83.57%`, Acc `75.10%`) | `+48.22 pts Esp.` |
| **Etapa 2 Condicional ($n=140$)** | Cardio vs. Derrame (solo patológicos) | — | **ROC-AUC `0.9521` (Acc. cond. `88.00%`, F1 `0.87`)** | — |

El desarrollo experimental permite extraer cinco conclusiones principales:

1. **Impacto de la curación de etiquetas en catálogos generados por minería de texto:** En repositorios hospitalarios como *ChestX-ray8/14* (National Institutes of Health Clinical Center, 2017; Wang et al., 2017), filtrar una patología buscando únicamente su nombre en `Finding Labels` incorpora estudios con patologías combinadas (`Cardiomegaly|Effusion`). Asimismo, la categoría `No Finding` (`Sin hallazgos`) solo indica la ausencia de las afecciones relevadas por el sistema NLP y no equivale a una radiografía estrictamente normal. Excluir la co-ocurrencia entre `Cardiomegaly` y `Effusion` y aislar los registros por `Patient_ID` eliminó la supervisión contradictoria del clasificador inicial, elevando la precisión multiclase de `Cardiomegalia` a `0.81` y la de `Derrame Pleural` a `0.65`.
2. **Ajuste fino de bloques convolucionales profundos:** Mantener congelados todos los bloques de `DenseNet-121` (Huang et al., 2017) limitó la exactitud multiclase al `52.17%`. Descongelar `denseblock3` y `denseblock4` en PyTorch (Paszke et al., 2019) con tasas de aprendizaje diferenciadas (`1e-4` en las capas convolucionales y `1e-3` en el clasificador lineal) durante 40 épocas reguladas con `Dropout`, decaimiento $L_2$ y *label smoothing* permitió alcanzar un `69.08%` de exactitud global y un `ROC-AUC Macro` de `0.8654` sobre pacientes no vistos (Litjens et al., 2017).
3. **Adecuación del preprocesamiento a la anatomía torácica:** Prescindir del filtro de mediana previo al reescalado a $224 \times 224$ píxeles y suprimir la inversión horizontal aleatoria (`RandomHorizontalFlip`) durante el entrenamiento preservaron la orientación espacial de la silueta cardíaca y los bordes pleurales, duplicando la sensibilidad de la categoría `Sin hallazgos (0)` (`0.36 -> 0.72`).
4. **Alcance del análisis en dos etapas y calibración de umbrales:** En la Etapa 1 (`n = 249`), calibrar las probabilidades por temperatura ($T = 1.121$) y seleccionar los puntos de corte sobre la curva ROC de validación con *Scikit-learn* (Pedregosa et al., 2011) permitió recuperar la especificidad frente al criterio basado en F1: el umbral de tamizaje fijado para alcanzar $\ge 90\%$ de sensibilidad en validación (`corte = 0.35`) obtuvo en el conjunto de prueba una sensibilidad del **`83.57%`**, una especificidad del **`64.22%`** y una exactitud binaria del **`75.10%`**. En la Etapa 2, la evaluación condicional restringida a los `140` casos realmente patológicos alcanzó un `ROC-AUC` de **`0.9521`** y una exactitud condicional del **`88.00%`**, demostrando que la red distingue con claridad `Cardiomegalia` de `Derrame Pleural` y que la menor sensibilidad global de `Cardiomegalia` (`0.50` en tres vías) obedece a su solapamiento con la categoría `Sin hallazgos (0)`.
5. **Limitaciones experimentales:** Este modelo constituye un prototipo académico de investigación y **no representa una herramienta validada clínicamente**. Entre sus limitaciones principales se encuentran el submuestreo a `1.739` imágenes de solo tres categorías (frente a las 14 afecciones del catálogo completo del NIH, acotado por las restricciones de almacenamiento y memoria de Google Colaboratory), la reducción de resolución de $1024 \times 1024$ a $224 \times 224$ píxeles, el ruido inherente a las etiquetas extraídas por procesamiento de lenguaje natural sin confirmación por un comité de radiólogos y la evaluación sobre una única institución hospitalaria.



## Referencias Bibliográficas

- Huang, G., Liu, Z., Van Der Maaten, L., & Weinberger, K. Q. (2017). Densely connected convolutional networks. En *Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition (CVPR)* (pp. 4700–4708). IEEE. https://doi.org/10.1109/CVPR.2017.243
- Litjens, G., Kooi, T., Bejnordi, B. E., Setio, A. A. A., Ciompi, F., Ghafoorian, M., van der Laak, J. A. W. M., van Ginneken, B., & Sánchez, C. I. (2017). A survey on deep learning in medical image analysis. *Medical Image Analysis*, *42*, 60–88. https://doi.org/10.1016/j.media.2017.07.005
- National Institutes of Health Clinical Center. (2017). *NIH Chest X-rays (ChestX-ray14) dataset* [Conjunto de datos]. National Institutes of Health. https://nihcc.app.box.com/v/ChestXray-NIHCC
- Paszke, A., Gross, S., Massa, F., Lerer, A., Bradbury, J., Chanan, G., Killeen, T., Lin, Z., Gimelshein, N., Antiga, L., Desmaison, A., Kopf, A., Yang, E., DeVito, Z., Raison, M., Tejani, A., Chilamkurthy, S., Steiner, B., Fang, L., Bai, J., & Chintala, S. (2019). PyTorch: An imperative style, high-performance deep learning library. En H. Wallach, H. Larochelle, A. Beygelzimer, F. d'Alché-Buc, E. Fox, & R. Garnett (Eds.), *Advances in Neural Information Processing Systems* (Vol. 32, pp. 8024–8035). Curran Associates, Inc. https://proceedings.neurips.cc/paper/2019/hash/bdbca288fee7f92f2bfa9f7012727740-Abstract.html
- Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M., & Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, *12*(85), 2825–2830. http://jmlr.org/papers/v12/pedregosa11a.html
- Wang, X., Peng, Y., Lu, L., Lu, Z., Bagheri, M., & Summers, R. M. (2017). ChestX-ray8: Hospital-scale chest X-ray database and benchmarks on weakly-supervised classification and localization of common thorax diseases. En *Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition (CVPR)* (pp. 2097–2106). IEEE. https://doi.org/10.1109/CVPR.2017.369